# Voz_Noslen F5-TTS ONNX (Modo Turbo) - v2026.06.17

Este notebook automatiza a criação do pacote **Turbo** para o F5-TTS, otimizado para CPU e Cloud Run.

### Estrutura do Pacote Gerado
```text
/onnx_package_turbo_TIMESTAMP/
├── onnx/f5_tts_transformer_core.onnx  # Grafo Turbo (x, cond -> dx)
├── model/vocab.txt                   # Dicionário de tokens
├── reference/referencia_voz.wav      # Áudio de referência
├── manifest.json                     # Versão e lista de arquivos
├── metadata.json                     # Contrato ONNX e Shapes
└── validation.json                   # Status da exportação e testes
```

**Requisitos:** Internet ativada no Kaggle.

In [ ]:
# 1) Instalação de dependências
!pip install -q huggingface_hub f5-tts safetensors onnx onnxruntime vocos

In [ ]:
# 2) Criação do Script Packager Turbo
from pathlib import Path
import json

script_content = r'''from __future__ import annotations

import argparse
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

PACKAGER_VERSION = "2026.06.17.turbo.v2"
DEFAULT_SOURCE_URL = "warllem/Voz_Noslen"
DEFAULT_VOICE_DIR = "voices/v_minha_voz_f5_tts_ptbr"

WORK_ROOT = Path("/kaggle/working")
DOWNLOAD_DIR = WORK_ROOT / "turbo_source_snapshot"
STAGING_DIR = WORK_ROOT / "turbo_staging_area"

@dataclass(frozen=True)
class TurboPaths:
    source: Path; staging: Path; onnx: Path; model: Path; reference: Path
    manifest: Path; metadata: Path; validation: Path

def setup_logging():
    logger = logging.getLogger("turbo_packager")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
        logger.addHandler(handler)
    return logger

LOGGER = setup_logging()

def clean_and_make_paths():
    if STAGING_DIR.exists(): shutil.rmtree(STAGING_DIR)
    paths = TurboPaths(
        source=DOWNLOAD_DIR, staging=STAGING_DIR, onnx=STAGING_DIR / "onnx",
        model=STAGING_DIR / "model", reference=STAGING_DIR / "reference",
        manifest=STAGING_DIR / "manifest.json", metadata=STAGING_DIR / "metadata.json",
        validation=STAGING_DIR / "validation.json"
    )
    for p in [paths.onnx, paths.model, paths.reference]: p.mkdir(parents=True, exist_ok=True)
    return paths

def copy_readonly(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

def export_turbo_core(checkpoint_path, vocab_path, paths):
    import torch
    from f5_tts.infer.utils_infer import load_model
    from f5_tts.model import DiT
    
    class F5TTSTurboWrapper(torch.nn.Module):
        def __init__(self, model): 
            super().__init__()
            self.transformer = getattr(model, "transformer", model)
        def forward(self, x, cond, text_ids, text_lengths, time_steps):
            return self.transformer(x, cond, text_ids, time_steps, text_lengths)

    arch_cfg = {"dim": 1024, "depth": 22, "heads": 16, "ff_mult": 2, "text_dim": 512, "text_mask_padding": True, "conv_layers": 4, "attn_backend": "torch"}
    onnx_file = paths.onnx / "f5_tts_transformer_core.onnx"
    model = load_model(DiT, arch_cfg, str(checkpoint_path), mel_spec_type="vocos", vocab_file=str(vocab_path), device="cpu")
    wrapped = F5TTSTurboWrapper(model).eval()
    example_inputs = (torch.randn(1, 128, 100), torch.randn(1, 128, 100), torch.randint(0, 100, (1, 64)), torch.tensor([64]), torch.tensor([0.5]))
    
    torch.onnx.export(
        wrapped, example_inputs, str(onnx_file),
        input_names=["x", "cond", "text_ids", "text_lengths", "time_steps"], output_names=["dx"],
        dynamic_axes={"x": {1: "duration"}, "cond": {1: "duration"}, "text_ids": {1: "text_len"}, "dx": {1: "duration"}},
        opset_version=17, do_constant_folding=True
    )

def main():
    paths = clean_and_make_paths()
    v_dir = DOWNLOAD_DIR / DEFAULT_VOICE_DIR
    ckpt = next(v_dir.glob("model/model_*.pt"))
    vocab = v_dir / "model/vocab.txt"
    ref = v_dir / "data_reference/referencia_voz.wav"
    
    copy_readonly(vocab, paths.model / "vocab.txt")
    copy_readonly(ref, paths.reference / "referencia_voz.wav")
    export_turbo_core(ckpt, vocab, paths)
    
    metadata = {"project": "Voz_Noslen Turbo", "contract": {"inputs": ["x", "cond", "text_ids", "text_lengths", "time_steps"], "outputs": ["dx"], "opset": 17, "dtype": "float32"}}
    paths.metadata.write_text(json.dumps(metadata, indent=2))
    
    manifest = {"version": PACKAGER_VERSION, "backend": "turbo-onnx", "files": {"onnx": "onnx/f5_tts_transformer_core.onnx", "vocab": "model/vocab.txt", "reference": "reference/referencia_voz.wav"}}
    paths.manifest.write_text(json.dumps(manifest, indent=2))
    
    LOGGER.info("PACOTE TURBO GERADO COM SUCESSO.")

if __name__ == "__main__": main()
'''

with open("f5_tts_onnx_packager_kaggle.py", "w") as f:
    f.write(script_content)
print("Script Turbo Packager criado com sucesso.")

In [ ]:
# 3) Download dos Ativos Originais (Read-Only)
import os
from huggingface_hub import snapshot_download

SOURCE_URL = "warllem/Voz_Noslen" 
DOWNLOAD_PATH = "/kaggle/working/turbo_source_snapshot"

hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        try:
            hf_token = user_secrets.get_secret("HF_TOKEN")
        except Exception:
            hf_token = user_secrets.get_secret("hf_token")
    except Exception as e:
        print(f"Aviso: Kaggle Secrets nao disponivel ou nao habilitado ({e}).")

if hf_token:
    print(f"[CHECK] Token HF encontrado (comprimento: {len(hf_token)}). Repositorio: {SOURCE_URL}")
else:
    print(f"[ALERTA] Nenhum token HF encontrado. Se o repo '{SOURCE_URL}' for privado, o download falhara.")
    print("DICA: Va em 'Add-ons' -> 'Secrets' no Kaggle e verifique se 'HF_TOKEN' esta marcado como habilitado para este notebook.")

print("Baixando ativos do Hugging Face...")
try:
    print("[INFO] Tentando download com repo_type='model'...")
    snapshot_download(
        repo_id=SOURCE_URL,
        local_dir=DOWNLOAD_PATH,
        allow_patterns=["voices/v_minha_voz_f5_tts_ptbr/*"],
        token=hf_token,
        repo_type="model"
    )
except Exception as model_err:
    if "404" in str(model_err) or "RepositoryNotFound" in model_err.__class__.__name__:
        print("[AVISO] Nao encontrado como 'model'. Tentando download com repo_type='dataset'...")
        try:
            snapshot_download(
                repo_id=SOURCE_URL,
                local_dir=DOWNLOAD_PATH,
                allow_patterns=["voices/v_minha_voz_f5_tts_ptbr/*"],
                token=hf_token,
                repo_type="dataset"
            )
        except Exception as dataset_err:
            print(f"[ERRO] Falha ao baixar como dataset: {dataset_err}")
            raise dataset_err
    else:
        print(f"[ERRO] Falha ao baixar como modelo: {model_err}")
        raise model_err
print(f"Snapshot concluído em: {DOWNLOAD_PATH}")

In [ ]:
# 4) Execução da Conversão e Empacotamento
!python f5_tts_onnx_packager_kaggle.py

In [ ]:
# 5) Compactação Final do Pacote
import shutil
import os
from datetime import datetime

STAGING_DIR = "/kaggle/working/turbo_staging_area"
zip_name = f"voz_noslen_turbo_{datetime.now().strftime('%Y%m%d_%H%M')}"

if os.path.exists(STAGING_DIR):
    shutil.make_archive(zip_name, 'zip', STAGING_DIR)
    print(f"PACOTE TURBO PRONTO: {zip_name}.zip")
else:
    print("ERRO: Pasta de staging não encontrada.")